# Day 43: Final Project Testing

YBI Foundation — Python AI & Data Science Internship

**Project:** AI-Based Student Performance Prediction System

## Objective

Rerun application notebooks and verify the complete regression suite against the final serving code.

## Theory

Final integration checks verify that interfaces, saved artifacts and notebooks still agree after accumulated changes. Edge cases include very large integers and CSV rows whose field counts differ from the header.

## Prerequisites

Install `requirements-dev.txt` (and `requirements-api.txt` for API work) in the active Python environment. Use the repository checkout and the saved Day 17 model. The setup below locates or clones the repository for Colab. Run cells in order.


In [1]:
from pathlib import Path
import os, sys, subprocess, json
root = Path.cwd()
if not (root / "src").exists() and (root.parent / "src").exists():
    root = root.parent
if not (root / "src").exists():
    target = root / "IYB_INTERNSHIP"
    if not target.exists():
        subprocess.run(["git", "clone", "https://github.com/lightcode01-oss/IYB_INTERNSHIP.git", str(target)], check=True)
    root = target
os.chdir(root)
sys.path.insert(0, str(root))
print("Project:", root.name)


Project: IYB_INTERNSHIP


Verify the successful fresh-kernel reruns of Days 19–42.

In [2]:
records=[json.loads(s) for s in Path("reports/notebook_execution.jsonl").read_text().splitlines()]
latest={Path(r["file"]).name:r for r in records}
from scripts.notebook_index import notebook_index
entries=notebook_index()
for day_number,path in entries.items():
    if day_number<=42:
        assert latest[Path(path).name]["status"]=="passed",path
print("Days 1–42 have successful recorded fresh-kernel execution; Days 19–42 rerun against the final application code.")

Days 1–42 have successful recorded fresh-kernel execution; Days 19–42 rerun against the final application code.


Run the final suite and preserve machine-readable test evidence.

In [3]:
r=subprocess.run([sys.executable,"-m","pytest","-q","--junitxml=reports/final_test_results.xml"],capture_output=True,text=True)
print(r.stdout)
Path("reports/FINAL_TEST_REPORT.md").write_text("# Final testing report\n\nExecuted command: python -m pytest -q --junitxml=reports/final_test_results.xml\n\n```text\n"+r.stdout+"\n```\n\nScope: application form submission, API/CSV parity, strict schema, malformed inputs, payload limits, model checksum/cache identity, monitoring concurrency, real artifact loading, explanation consistency and daily notebook execution state. Local HTTP startup was separately verified on Day 39. Tests use public UCI records or explicit synthetic test cases in temporary storage.\n\nKnown upstream deprecation warnings are reported in the output. This suite is not a production penetration test or educational validation.")
assert r.returncode==0,r.stderr

........................................................................ [ 84%]
.............                                                            [100%]
============================== warnings summary ===============================
..\venv\Lib\site-packages\fastapi\testclient.py:1
  C:\Users\ABHINAV DASH\Documents\Codex\2026-11-02\and\work\venv\Lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
    from starlette.testclient import TestClient as TestClient  # noqa

tests/test_app_integration.py::test_single_student_submission
tests/test_app_integration.py::test_single_student_submission
tests/test_deployment_readiness.py::TestDeploymentReadiness::test_08_streamlit_app_test_execution
  C:\Users\ABHINAV DASH\Documents\Codex\2026-11-02\and\work\venv\Lib\site-packages\streamlit\elements\lib\built_in_chart_utils.py:620: Pandas4Warning: The inplace keyword in DataFrame.reset_index i

Validate all daily notebook structures and source syntax.

In [4]:
import nbformat
for path in entries.values():
    nbformat.validate(nbformat.read(path,as_version=4))
r=subprocess.run([sys.executable,"-m","compileall","-q","src","scripts","pages","app.py","api.py"],capture_output=True,text=True)
assert r.returncode==0,r.stderr
print("Notebook schema and source syntax: PASS")

Notebook schema and source syntax: PASS


## Findings and interpretation

Recorded fresh-kernel reruns and the final regression output provide reviewable evidence. Very large numeric inputs and ragged CSV rows now return validation errors before inference.

## Limitations

Passing tests cover the stated cases and local application behavior. They do not establish external-school generalization, production security or a hosted deployment.

## Development log

Implemented and ran the cells above during the repair and extension session on 2 November 2026. The saved outputs are produced by the kernel.

## Conclusion

Rerun application notebooks and verify the complete regression suite against the final serving code.

## Next-day plan

Prepare the portfolio README, demonstration instructions and an actual app screenshot.